# TOBi Routing & Escalation - Executive Dashboard

**Audience:** management & leadership.  
**Purpose:** explain, in plain language with visuals, how often technical
requests are misrouted into non-technical chat queues, what it costs, why it
happens, and what to do about it.

This notebook is the narrative companion to the SQL in `models/` and
`analysis/`. Every chart is preceded by a short explanation and the business
question it answers.

> **Run order:** build the views first in BigQuery (`models/01` -> `02` -> `03`
> -> `04`), then run this notebook top to bottom. All numbers come from the two
> source tables (`f_kafka_tobi_sessions` + `f_tobi_logs_vertex`) via the
> `v_session_master` view.

## How to read this dashboard

- **Red** = misroute / problem.  **Green** = resolved by the bot.  **Blue** = correctly routed.
- A *technical* request = TV/phone features, connection problems, or phone damage/repair.
- A *hard misroute* = a technical request that was transferred to a **non-technical** queue (e.g. sales/CPOS).

## 0. Setup & data load

In [ ]:
# %pip install google-cloud-bigquery pandas db-dtypes matplotlib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from google.cloud import bigquery

# ---- brand theme --------------------------------------------------------
RED='#E60000'; DARK='#25282A'; GREY='#7E8083'; LGREY='#E9EAEC'
GREEN='#009900'; AMBER='#FBA600'; BLUE='#0077C8'; INK='#4A4D4E'
plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'white','axes.edgecolor':LGREY,
    'axes.grid':True,'grid.color':LGREY,'grid.linewidth':0.9,'axes.axisbelow':True,
    'axes.spines.top':False,'axes.spines.right':False,'axes.titlesize':14,
    'axes.titleweight':'bold','axes.titlecolor':DARK,'font.size':11,'figure.dpi':110})
TOPIC_LABEL={'connection_problem':'Connection problems',
             'device_damage_repair':'Phone damage / repair',
             'feature_question':'TV / phone features'}

def bar_labels(ax,bars,fmt='{:.0f}',pad=3,color=DARK):
    for b in bars:
        h=b.get_height()
        ax.annotate(fmt.format(h),(b.get_x()+b.get_width()/2,h),ha='center',
                    va='bottom',xytext=(0,pad),textcoords='offset points',
                    fontsize=10,fontweight='bold',color=color)

In [ ]:
PROJECT='vf-pt-copsvertex-live'
ANALYSIS_DS='tobi_routing_analysis'    # dataset where the model views live
client=bigquery.Client(project=PROJECT)
MASTER=f'`{PROJECT}.{ANALYSIS_DS}.v_session_master`'

df=client.query(f'''
SELECT START_MOMENT, CHANNEL, confidence_band, technical_topic_type,
       is_technical_topic, routed_queue_category, routed_queue_subtype,
       final_transfer_target, was_transferred, n_transfers, duration_seconds,
       is_hard_misroute, is_soft_misroute, is_correct_technical_route,
       is_fcr, repeat_contact_24h
FROM {MASTER}
''').to_dataframe()

tech=df[df.is_technical_topic]
print(f'{len(df):,} sessions  |  {len(tech):,} technical')

## 1. Executive summary (KPI cards)

The four numbers leadership should remember: how big the technical workload is,
how much of it goes to the wrong place, how much the bot resolves on its own,
and the agent-time we could recover by fixing routing.

In [ ]:
n_tech=len(tech)
hard_pct=100*tech.is_hard_misroute.mean()
fcr_pct=100*tech.is_fcr.mean()
excess_s=(tech[tech.is_hard_misroute].duration_seconds.mean()
          -tech[tech.is_correct_technical_route].duration_seconds.mean())
hours=excess_s*tech.is_hard_misroute.sum()/3600

cards=[('Technical sessions',f'{n_tech:,}','in analysis window',INK),
       ('Misrouted to wrong queue',f'{hard_pct:.0f}%','of technical requests',RED),
       ('First-contact resolution',f'{fcr_pct:.0f}%','technical, bot-contained',GREEN),
       ('Recoverable agent-hours',f'{hours:,.0f}','if misroutes fixed',BLUE)]

fig,axes=plt.subplots(1,4,figsize=(15,2.6))
for ax,(t,v,s,c) in zip(axes,cards):
    ax.axis('off')
    ax.add_patch(FancyBboxPatch((0.04,0.08),0.92,0.84,
        boxstyle='round,pad=0.02,rounding_size=0.05',lw=0,fc=LGREY,transform=ax.transAxes))
    ax.add_patch(FancyBboxPatch((0.04,0.08),0.03,0.84,boxstyle='square,pad=0',
        lw=0,fc=c,transform=ax.transAxes))
    ax.text(0.14,0.62,v,fontsize=24,fontweight='bold',color=c,transform=ax.transAxes,va='center')
    ax.text(0.14,0.30,t,fontsize=10.5,fontweight='bold',color=DARK,transform=ax.transAxes,va='center')
    ax.text(0.14,0.15,s,fontsize=8.5,color=GREY,transform=ax.transAxes,va='center')
plt.tight_layout(); plt.show()

## 2. Deliverable 1 - Where misrouting happens

**Question:** which technical topics are most often sent to the wrong queue, and
where do they end up? The left chart shows volume + rate by topic; the right
chart shows the non-technical queues that absorb misrouted technical issues.

In [ ]:
g=(tech.groupby('technical_topic_type').is_hard_misroute.agg(['size','sum']))
g['pct']=100*g['sum']/g['size']; g=g.sort_values('sum')
land=df[df.is_hard_misroute].groupby('routed_queue_subtype').size().sort_values()
land.index=[i.replace('_',' ').title() for i in land.index]

fig,ax=plt.subplots(1,2,figsize=(14,4.4))
b=ax[0].barh([TOPIC_LABEL.get(i,i) for i in g.index],g['sum'],color=RED)
for bar,p in zip(b,g.pct):
    ax[0].annotate(f'{int(bar.get_width()):,} ({p:.0f}%)',(bar.get_width(),bar.get_y()+bar.get_height()/2),
                   xytext=(5,0),textcoords='offset points',va='center',fontweight='bold',fontsize=9)
ax[0].set_title('Misrouting by technical topic'); ax[0].margins(x=0.22)
b2=ax[1].barh(land.index,land.values,color=RED)
for bar in b2:
    ax[1].annotate(f'{int(bar.get_width()):,}',(bar.get_width(),bar.get_y()+bar.get_height()/2),
                   xytext=(5,0),textcoords='offset points',va='center',fontweight='bold')
ax[1].set_title('Where misrouted issues land (non-technical queues)'); ax[1].margins(x=0.18)
plt.tight_layout(); plt.show()

## 3. Deliverable 2 - The cost of misrouting

**Question:** what does misrouting cost the customer and the operation? Left:
average handling time by outcome - misrouted sessions take far longer. Right:
how often customers have to come back within 24 hours.

In [ ]:
s=pd.Series({'Bot resolved':tech[tech.is_fcr].duration_seconds.mean(),
             'Correctly routed':tech[tech.is_correct_technical_route].duration_seconds.mean(),
             'Misrouted':tech[tech.is_hard_misroute].duration_seconds.mean()})/60
r=pd.Series({'Correctly routed':100*tech[tech.is_correct_technical_route].repeat_contact_24h.mean(),
             'Misrouted':100*tech[tech.is_hard_misroute].repeat_contact_24h.mean()})

fig,ax=plt.subplots(1,2,figsize=(14,4.6))
b=ax[0].bar(s.index,s.values,color=[GREEN,BLUE,RED],width=.6); bar_labels(ax[0],b,'{:.1f} min')
ax[0].set_title('Average handling time'); ax[0].set_ylabel('minutes'); ax[0].set_ylim(0,s.max()*1.25)
b2=ax[1].bar(r.index,r.values,color=[BLUE,RED],width=.5); bar_labels(ax[1],b2,'{:.0f}%')
ax[1].set_title('Repeat contact within 24h'); ax[1].set_ylabel('%'); ax[1].set_ylim(0,r.max()*1.3)
plt.tight_layout(); plt.show()

## 4. Deliverable 3 - Why it happens (root cause)

**Question:** what drives the wrong routing decision? Left: misroute rate rises
sharply as intent-detection **confidence** falls - the bot misreads the request.
Right: some **entry channels** misroute more than others.

In [ ]:
order=[o for o in ['none','low','medium','high'] if o in tech.confidence_band.unique()]
gc=(tech.groupby('confidence_band').is_hard_misroute.mean().reindex(order)*100)
gch=(tech.groupby('CHANNEL').is_hard_misroute.mean()*100).sort_values(ascending=False)

fig,ax=plt.subplots(1,2,figsize=(14,4.4))
b=ax[0].bar(gc.index,gc.values,color=[GREY,RED,AMBER,GREEN][:len(gc)],width=.6); bar_labels(ax[0],b,'{:.0f}%')
ax[0].set_title('Misroute % by intent-detection confidence'); ax[0].set_ylim(0,gc.max()*1.25)
b2=ax[1].bar(gch.index,gch.values,color=INK,width=.6); bar_labels(ax[1],b2,'{:.0f}%')
ax[1].set_title('Misroute % by entry channel'); ax[1].set_ylim(0,gch.max()*1.25)
plt.tight_layout(); plt.show()

## 5. Deliverable 5 - End-to-end fate of technical requests

**Question:** of all technical requests, how many does the bot resolve, how many
are correctly routed, and how many are misrouted? This funnel is the single
clearest picture of the containment and routing opportunity.

In [ ]:
total=n_tech
stages=[('Technical requests',total,INK),
        ('Bot resolved (FCR)',int(tech.is_fcr.sum()),GREEN),
        ('Correctly routed',int(tech.is_correct_technical_route.sum()),BLUE),
        ('Misrouted',int(tech.is_hard_misroute.sum()),RED)]
fig,ax=plt.subplots(figsize=(11,4))
for i,(lab,val,col) in enumerate(stages):
    ax.barh(i,val,color=col,height=.62)
    ax.annotate(f'{val:,} ({100*val/total:.0f}%)',(val,i),xytext=(8,0),
                textcoords='offset points',va='center',fontweight='bold')
ax.set_yticks(range(len(stages))); ax.set_yticklabels([s[0] for s in stages])
ax.invert_yaxis(); ax.set_title('What happens to technical requests'); ax.margins(x=0.2)
plt.tight_layout(); plt.show()

## 6. Trend over time

**Question:** is misrouting getting better or worse? The 7-day average smooths
daily noise so leadership can see the direction of travel.

In [ ]:
t=tech.copy(); t['day']=pd.to_datetime(t.START_MOMENT).dt.date
daily=t.groupby('day').is_hard_misroute.agg(['size','sum'])
daily['pct']=100*daily['sum']/daily['size']
roll=daily.pct.rolling(7,min_periods=1).mean()
fig,ax=plt.subplots(figsize=(12,4))
ax.plot(daily.index,daily.pct,color=LGREY,lw=1.2,label='Daily')
ax.plot(daily.index,roll,color=RED,lw=2.6,label='7-day average')
ax.set_title('Technical misroute rate over time'); ax.set_ylabel('%')
ax.legend(frameon=False); fig.autofmt_xdate(); plt.tight_layout(); plt.show()

## 7. Deliverable 4 - Prioritised actions

**Question:** where do we act first? Each row is an (intent -> wrong destination)
leak ranked by an impact score (volume + 2x repeat contacts + handovers). The top
rows are the highest-ROI routing-rule fixes.

In [ ]:
mis=tech[tech.is_hard_misroute]
leaks=(mis.groupby(['technical_topic_type','final_transfer_target'])
          .agg(misrouted=('duration_seconds','size'),
               repeats=('repeat_contact_24h','sum'),
               handovers=('n_transfers','sum')).reset_index())
leaks['impact_score']=leaks.misrouted+2*leaks.repeats+leaks.handovers
leaks.sort_values('impact_score',ascending=False).head(15)

## 8. Recommendations & next steps

1. **Re-point routing rules** for the top leaks above (technical intents wired to
   non-technical queues).
2. **Improve intent detection** for low-confidence technical requests (training
   phrases / disambiguation) - this is the largest root-cause driver.
3. **Add a technical off-ramp** in the flows/channels with the highest misroute %.
4. **Track the KPI scorecard** (`analysis/d4` 4c) weekly to confirm FCR uplift and
   misroute reduction.

---
*All figures derive solely from `f_kafka_tobi_sessions` + `f_tobi_logs_vertex`.
Numbers depend on the queue classification (`models/02`) and technical-topic
keywords (`models/03`) - validate those against the real `T_`/`R_`/`M_` vocabulary
before circulating externally.*